## Node components — `kubelet`, `kube-proxy`, the container runtime

Every node — control-plane or worker — runs three things in coordination.

**`kubelet`** — the node agent. Crucially, **it does not run as a pod** — it's a systemd service. Its job:

1. Register the node with the API server.
2. Watch for pods with `spec.nodeName == <this node>`.
3. Drive the runtime to make each real — pull image, create the network namespace, mount volumes, start containers.
4. Run **probes** (notebook 02); restart per `restartPolicy`.
5. Report status back — phases, conditions, container states.
6. Watch node pressure and **evict** over thresholds.

**The kubelet does not schedule** — it only runs what's been *bound* to its node.

**`kube-proxy`** — the data-plane half of Services. Watches Services + EndpointSlices, programs the node's kernel networking (`iptables`/`IPVS`/`nftables` — notebook 04) so ClusterIPs route to pod IPs. Runs as a DaemonSet. Optional now — some CNIs (Cilium with eBPF) replace it.

**Container runtime** — what actually runs containers. Kubernetes ships none; it speaks **CRI** to whatever's installed: **`containerd`** (most common) or **`CRI-O`**. Docker Engine is no longer supported — the `dockershim` bridge was removed in 1.24. The runtime sits at a socket like `/run/containerd/containerd.sock`, named by the kubelet's `--container-runtime-endpoint`.

On our map these three *are* the **Node runtime** box — **kubelet**, **kube-proxy**, and **CRI (containerd)** — the "execute" half that turns the control plane's decisions into running Pods.